# Coverage-Guided Constraint Fuzzing for RISC Zero

## Executive Summary

**What changed:** From random mutation scheduling to a **Discounted-UCB bandit scheduler** that uses constraint-coverage feedback (touch, failure-context, Z events) to guide mutation selection.

**What we can now measure per mutation:**
- **Touched constraints** (local): which constraint sites were active during witness generation
- **Failing constraint contexts** (local): which `(constraint_loc, major, minor)` triples reported non-zero residuals
- **Post-local rejection (Z events)**: mutations passing all local checks but rejected at verification — failure in non-instrumented checks (global constraints, permutation arguments)

**Campaigns compared:**

| | Uniform (baseline) | Bandit (B=32, 254 arms) |
|---|---|---|
| Selector | Zoned random (5%/90%/5%) | Discounted-UCB bandit |
| Budget | 1000 mutations | 1000 (50 pilot + 950 bandit) |
| Seed | 777 | 777 |

**Limitations:** Only local constraints instrumented. Z = "post-local rejection," not a specific global constraint ID.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
import math

from a4.standalone.tests.analyze_campaign import (
    parse_terminal, RunRecord,
    compute_cumulative_metrics, compute_cumulative_from_db,
    compute_auc_normalized, compute_t80,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
UNIFORM_TERMINAL = os.path.join(PROJECT_ROOT, 'uniform_1000_output.txt')
BANDIT32_TERMINAL = '/root/.cursor/projects/root-arguzz/terminals/633840.txt'
UNIFORM_DB = os.path.join(PROJECT_ROOT, 'uniform_baseline_1000.db')
BANDIT32_DB = os.path.join(PROJECT_ROOT, 'a4_coverage.db')
BANDIT32_CAMPAIGN_ID = 57

u_runs, u_meta = parse_terminal(UNIFORM_TERMINAL)
b_runs, b_meta = parse_terminal(BANDIT32_TERMINAL)

u_all = [r for r in u_runs if not r.is_pilot]
b_pilot = [r for r in b_runs if r.is_pilot]
b_bandit = [r for r in b_runs if not r.is_pilot]

u_df = pd.DataFrame([vars(r) for r in u_all])
u_df['idx'] = range(len(u_df))
b_df = pd.DataFrame([vars(r) for r in b_bandit])
b_df['idx'] = range(len(b_df))

SHORT = {
    'COMP_OUT_MOD': 'COMP', 'LOAD_VAL_MOD': 'LOAD', 'STORE_OUT_MOD': 'STORE',
    'PRE_EXEC_REG_MOD': 'PRE_REG', 'INSTR_TYPE_MOD': 'ITYPE',
    'MEM_VAL_MOD': 'MEM', 'INSTR_WORD_MOD_FULL': 'IWORD_F',
    'INSTR_WORD_MOD_SUR': 'IWORD_S',
}
u_df['kind_short'] = u_df['kind'].map(SHORT)
b_df['kind_short'] = b_df['kind'].map(SHORT)
KIND_ORDER = ['IWORD_S', 'IWORD_F', 'ITYPE', 'PRE_REG', 'COMP', 'MEM', 'LOAD', 'STORE']

u_db_cm = compute_cumulative_from_db(UNIFORM_DB)
b_db_cm = compute_cumulative_from_db(BANDIT32_DB, campaign_id=BANDIT32_CAMPAIGN_ID)

u_cm = compute_cumulative_metrics(u_all)
b_cm = compute_cumulative_metrics(b_bandit)

print(f'Uniform:   {len(u_all)} runs')
print(f'Bandit-32: {len(b_pilot)} pilot + {len(b_bandit)} bandit = {len(b_runs)} total')

---
## Section B: Did We Explore the Constraint Space Better?

Four cumulative coverage metrics: **Uniform (red dashed)** vs **Bandit-32 (blue solid)**. Failure-context and family metrics from SQLite (authoritative counts); Z and crashes from terminal.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Cumulative Coverage: Uniform vs Bandit', fontsize=16, fontweight='bold')

ax = axes[0, 0]
ax.plot(range(len(u_db_cm['cum_fail_contexts'])), u_db_cm['cum_fail_contexts'],
        'r--', lw=2, label=f'Uniform ({u_db_cm["cum_fail_contexts"][-1]})')
ax.plot(range(len(b_db_cm['cum_fail_contexts'])), b_db_cm['cum_fail_contexts'],
        'b-', lw=2, label=f'Bandit-32 ({b_db_cm["cum_fail_contexts"][-1]})')
ax.set_title('Cumulative Distinct Failure Contexts')
ax.set_xlabel('Mutation #'); ax.set_ylabel('Distinct (loc, major, minor)'); ax.legend()

ax = axes[0, 1]
ax.plot(range(len(u_db_cm['cum_families'])), u_db_cm['cum_families'],
        'r--', lw=2, label=f'Uniform ({u_db_cm["cum_families"][-1]})')
ax.plot(range(len(b_db_cm['cum_families'])), b_db_cm['cum_families'],
        'b-', lw=2, label=f'Bandit-32 ({b_db_cm["cum_families"][-1]})')
ax.set_title('Cumulative Distinct Constraint Families')
ax.set_xlabel('Mutation #'); ax.set_ylabel('Distinct families'); ax.legend()

ax = axes[1, 0]
ax.plot(u_df['idx'], np.cumsum(u_df['Z']),
        'r--', lw=2, label=f'Uniform ({int(u_df["Z"].sum())})')
ax.plot(b_df['idx'], np.cumsum(b_df['Z']),
        'b-', lw=2, label=f'Bandit-32 ({int(b_df["Z"].sum())})')
ax.set_title('Cumulative Z Events (Post-Local Rejections)')
ax.set_xlabel('Run #'); ax.set_ylabel('Z events'); ax.legend()

ax = axes[1, 1]
ax.plot(u_df['idx'], np.cumsum((u_df['outcome']=='CRASH').astype(int)),
        'r--', lw=2, label=f'Uniform ({int((u_df["outcome"]=="CRASH").sum())})')
ax.plot(b_df['idx'], np.cumsum((b_df['outcome']=='CRASH').astype(int)),
        'b-', lw=2, label=f'Bandit-32 ({int((b_df["outcome"]=="CRASH").sum())})')
ax.set_title('Cumulative Crashes')
ax.set_xlabel('Run #'); ax.set_ylabel('Crashes'); ax.legend()

plt.tight_layout()
plt.show()

rows = []
for name, uc, bc in [
    ('Failure contexts', u_db_cm['cum_fail_contexts'], b_db_cm['cum_fail_contexts']),
    ('Constraint families', u_db_cm['cum_families'], b_db_cm['cum_families']),
    ('Z events', list(np.cumsum(u_df['Z']).astype(int)), list(np.cumsum(b_df['Z']).astype(int))),
]:
    rows.append({'Metric': name,
        'Uni final': uc[-1], 'Ban final': bc[-1],
        'Uni AUC': round(compute_auc_normalized(uc),3),
        'Ban AUC': round(compute_auc_normalized(bc),3),
        'Uni t80': compute_t80(uc), 'Ban t80': compute_t80(bc)})
print(pd.DataFrame(rows).to_string(index=False))

---
## Section C: What Did the Bandit Learn?

The reward function identifies **structural differences** between mutation kinds. Two high-value regimes emerge: instruction-type mutations (touch/failure novelty) and instruction-word mutations (Z events).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=u_df, x='kind_short', y='reward', order=KIND_ORDER,
            ax=axes[0], palette='Reds', showfliers=False)
axes[0].set_title('Uniform: Reward by Kind'); axes[0].set_ylabel('Reward')
axes[0].tick_params(axis='x', rotation=45)
sns.boxplot(data=b_df, x='kind_short', y='reward', order=KIND_ORDER,
            ax=axes[1], palette='Blues', showfliers=False)
axes[1].set_title('Bandit-32: Reward by Kind'); axes[1].set_ylabel('Reward')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

print(f'{"Kind":>10s}  {"Uniform":>8s}  {"Bandit":>8s}')
for k in KIND_ORDER:
    um = u_df[u_df['kind_short']==k]['reward'].mean()
    bm = b_df[b_df['kind_short']==k]['reward'].mean()
    print(f'{k:>10s}  {um:8.3f}  {bm:8.3f}')

In [ ]:
T, B_count = 3930, 32
B = math.ceil(T / B_count)
b_df['bucket'] = b_df['step'] // B

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
pivot_r = b_df.pivot_table(values='reward', index='kind_short', columns='bucket', aggfunc='mean').reindex(KIND_ORDER)
sns.heatmap(pivot_r, ax=axes[0], cmap='YlOrRd', annot=False, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Mean Reward'})
axes[0].set_title('Mean Reward per Arm'); axes[0].set_xlabel('Bucket'); axes[0].set_ylabel('Kind')

pivot_c = b_df.pivot_table(values='reward', index='kind_short', columns='bucket', aggfunc='count').reindex(KIND_ORDER)
sns.heatmap(pivot_c, ax=axes[1], cmap='Blues', annot=False, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Pulls'})
axes[1].set_title('Selection Count per Arm'); axes[1].set_xlabel('Bucket'); axes[1].set_ylabel('Kind')

arm_s = b_df.groupby(['kind_short','bucket']).agg(mean_reward=('reward','mean'),count=('reward','count')).reset_index()
axes[2].scatter(arm_s['mean_reward'], arm_s['count'], alpha=0.5, s=30, c='#2c3e50')
z = np.polyfit(arm_s['mean_reward'], arm_s['count'], 1)
axes[2].plot(np.linspace(0, arm_s['mean_reward'].max()*1.05, 100),
             np.poly1d(z)(np.linspace(0, arm_s['mean_reward'].max()*1.05, 100)),
             'r--', lw=2, label=f'slope={z[0]:.1f}')
rc = np.corrcoef(arm_s['mean_reward'], arm_s['count'])[0,1]
axes[2].set_title(f'Arm Reward vs Pulls (r={rc:.3f})'); axes[2].set_xlabel('Mean Reward')
axes[2].set_ylabel('Pulls'); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
arm_d = b_df.groupby(['kind_short','bucket']).agg(
    pulls=('reward','count'), mean_r=('reward','mean'),
    z_cnt=('Z','sum'), mean_nf=('n_fail','mean'), mean_Q=('Q','mean')).reset_index()
arm_d['z_rate'] = arm_d['z_cnt'] / arm_d['pulls']
top15 = arm_d.nlargest(15, 'mean_r')
print('=== TOP 15 ARMS BY MEAN REWARD ===')
print(f'{"Kind":>10s} {"Bkt":>4s} {"Pulls":>5s} {"Mean r":>7s} {"Z rate":>7s} {"nf":>5s} {"Q":>5s}')
for _, r in top15.iterrows():
    print(f'{r["kind_short"]:>10s} {int(r["bucket"]):4d} {int(r["pulls"]):5d} '
          f'{r["mean_r"]:7.3f} {r["z_rate"]:7.1%} {r["mean_nf"]:5.1f} {r["mean_Q"]:5.2f}')

---
## Section D: Why Z Events Matter

**Z events** = mutations with no local constraint failures, yet proof rejected. Failure is in non-instrumented logic (likely global constraints).

For soundness-bug hunting, Z events are closest to the acceptance boundary.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

ax = axes[0]
x = np.arange(len(KIND_ORDER)); w = 0.35
for i, (lbl, df, c) in enumerate([('Uniform', u_df, '#e74c3c'), ('Bandit-32', b_df, '#3498db')]):
    zk = df.groupby('kind_short')['Z'].sum().reindex(KIND_ORDER, fill_value=0)
    tk = df.groupby('kind_short').size().reindex(KIND_ORDER, fill_value=1)
    ax.bar(x + (-w/2 if i==0 else w/2), zk/tk*100, w, label=lbl, color=c, alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(KIND_ORDER, rotation=45)
ax.set_title('Z Rate by Kind'); ax.set_ylabel('Z rate (%)'); ax.legend()

ax = axes[1]
bz = b_df.copy(); bz['bucket'] = bz['step'] // math.ceil(3930/32)
pz = bz.pivot_table(values='Z', index='kind_short', columns='bucket', aggfunc='mean').reindex(KIND_ORDER)*100
sns.heatmap(pz, ax=ax, cmap='YlOrRd', annot=False, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Z rate (%)'})
ax.set_title('Z Rate per Arm (Bandit)'); ax.set_xlabel('Bucket'); ax.set_ylabel('Kind')

ax = axes[2]
bins = np.linspace(0, 4000, 30)
ax.hist(u_df[u_df['Z']==1]['step'], bins=bins, alpha=0.5, color='#e74c3c', label=f'Uniform ({int(u_df["Z"].sum())})')
ax.hist(b_df[b_df['Z']==1]['step'], bins=bins, alpha=0.5, color='#3498db', label=f'Bandit ({int(b_df["Z"].sum())})')
ax.set_title('Z Events by Step Region'); ax.set_xlabel('Step'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout(); plt.show()

uz, bz_ = int(u_df['Z'].sum()), int(b_df['Z'].sum())
print(f'{"":20s} {"Uniform":>10s} {"Bandit-32":>10s}')
print(f'{"Z events":20s} {uz:10d} {bz_:10d}')
print(f'{"Z rate":20s} {uz/len(u_df)*100:9.1f}% {bz_/len(b_df)*100:9.1f}%')
uzr = u_df[u_df['Z']==1]['reward'].mean(); bzr = b_df[b_df['Z']==1]['reward'].mean()
unr = u_df[(u_df['Z']==0)&(u_df['outcome']!='CRASH')]['reward'].mean()
bnr = b_df[(b_df['Z']==0)&(b_df['outcome']!='CRASH')]['reward'].mean()
print(f'{"Z mean reward":20s} {uzr:10.3f} {bzr:10.3f}')
print(f'{"Non-Z mean reward":20s} {unr:10.3f} {bnr:10.3f}')
print(f'{"Z / non-Z ratio":20s} {uzr/max(unr,1e-9):9.1f}x {bzr/max(bnr,1e-9):9.1f}x')

---
## Section E: Quality Control — Crashes and Cascades

The Q multiplier penalizes crashes (Q=0) and cascade failures. This prevents the bandit from chasing garbage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]; ax.axis('off')
cd = []
for lbl, df in [('Uniform', u_df), ('Bandit-32', b_df)]:
    cr = df[df['outcome']=='CRASH']
    if len(cr) > 0:
        for k in cr['kind'].unique():
            kc = cr[cr['kind']==k]
            cd.append([lbl, SHORT.get(k,k), len(kc), ', '.join(str(s) for s in sorted(kc['step'].unique())[:5])])
if cd:
    t = ax.table(cellText=cd, colLabels=['Campaign','Kind','Count','Steps'], loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.5)
ax.set_title('Crashes', fontweight='bold', pad=20)

ax = axes[1]; ax.axis('off')
cr2 = []
for lbl, df in [('Uniform', u_df), ('Bandit-32', b_df)]:
    tc = df[df['n_fail']>10].nlargest(3, 'n_fail')
    for _, r in tc.iterrows():
        cr2.append([lbl, r['kind_short'], int(r['n_fail']), f'{r["reward"]:.3f}'])
if cr2:
    t = ax.table(cellText=cr2, colLabels=['Campaign','Kind','n_fail','Reward'], loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.5)
ax.set_title('Top Cascades (n_fail > 10)', fontweight='bold', pad=20)
plt.tight_layout(); plt.show()

---
## Key Takeaways

1. **We built a measurable coverage signal for zkVM constraints** — touched local constraints, failing constraint contexts, and post-local rejection (Z events).

2. **The reward function identifies two high-value mutation regimes:**
   - Instruction-type mutations unlock new constraint paths (high touch/failure novelty)
   - Instruction-word mutations pass all local checks but are rejected later (Z events: ~10% rate, ~4x reward multiplier)

3. **At 1000 mutations with 254 arms, coverage rates are comparable.** The bandit discovers more constraint families but hasn't had enough samples/arm (~3.7) to strongly exploit. The reduced-arm campaign (128 arms) should show clearer exploitation.

4. **Z events are the most promising signal for soundness bugs.** Exclusively from instruction-word mutations. Represent mutations closest to acceptance.

### Next Steps
- Reduced-arm bandit (B=16, 128 arms): demonstrate exploitation
- Z signature analysis (Phase III.0): cluster Z by rejection error
- Global constraint instrumentation (Phase III.1): replace Z with explicit global contexts

---
# Appendix: Algorithm Specification

## Reward: $r = \min(1, Q \cdot S)$

$$S = \frac{a_{T_n} T_{\text{new}} + a_{T_r} T_{\text{rare}} + a_{F_n} F_{\text{new}} + a_{F_r} F_{\text{rare}} + a_Z Z}{\sum a_i}$$

- $T_{\text{new}} = 1 - e^{-\Delta_T / \tau_T}$, $F_{\text{new}} = 1 - e^{-\Delta_F / \tau_F}$
- $T_{\text{rare}} = \frac{1}{K_T}\sum_{\text{top-}K_T} (1+f_T[i])^{-1/2}$, $F_{\text{rare}} = \frac{1}{K_F}\sum_{\text{top-}K_F} (1+f_F[c])^{-1/2}$
- $Z = \mathbb{1}[\text{REJECTED} \wedge \text{proof\_gen} \wedge d_{\text{fail}}=0]$

## Quality: $Q = Q_{\text{dist}} \times Q_{\text{rep}}$
- $Q_{\text{dist}} = e^{-d_{\text{fail}}/\tau_d}$, $Q_{\text{rep}} = \begin{cases}1 & r_{\text{rep}} \leq r_0 \\ e^{-(r_{\text{rep}}-r_0)/\tau_r} & \text{else}\end{cases}$

## Bandit: Discounted UCB
$$\text{UCB}_a = \hat{\mu}_a + c\sqrt{\frac{\ln N_{\text{tot}}}{N_a}}, \quad N_a \leftarrow N_a \gamma^{\Delta t}$$

Weights: $a_{T_n}\!=\!1, a_{T_r}\!=\!0.25, a_{F_n}\!=\!1, a_{F_r}\!=\!1, a_Z\!=\!1$. $c\!=\!0.25$, $\gamma\!\approx\!0.997$.

In [ ]:
print('=== CAMPAIGN COMPARISON ===')
print(f'{"":25s} {"Uniform":>10s} {"Bandit-32":>10s}')
print(f'{"Total mutations":25s} {len(u_all):10d} {len(b_runs):10d}')
print(f'{"  (pilot)":25s} {0:10d} {len(b_pilot):10d}')
print(f'{"  (main)":25s} {len(u_all):10d} {len(b_bandit):10d}')
print(f'{"REJECTED":25s} {len(u_df[u_df["outcome"]=="REJECTED"]):10d} {len(b_df[b_df["outcome"]=="REJECTED"]):10d}')
print(f'{"CRASH":25s} {len(u_df[u_df["outcome"]=="CRASH"]):10d} {len(b_df[b_df["outcome"]=="CRASH"]):10d}')
print(f'{"Distinct fail contexts":25s} {u_db_cm["cum_fail_contexts"][-1]:10d} {b_db_cm["cum_fail_contexts"][-1]:10d}')
print(f'{"Distinct families":25s} {u_db_cm["cum_families"][-1]:10d} {b_db_cm["cum_families"][-1]:10d}')
print(f'{"Z events":25s} {int(u_df["Z"].sum()):10d} {int(b_df["Z"].sum()):10d}')
print(f'{"Mean reward":25s} {u_df["reward"].mean():10.4f} {b_df["reward"].mean():10.4f}')